In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.rl.env_autoscale import AutoscaleEnv
from src.rl.agent_ppo import PPOAgent


In [ ]:
from src.rl.train_rl import train_rl
agent = train_rl(num_episodes=50)


In [ ]:
def evaluate(agent, episodes=50):
    env = AutoscaleEnv()
    returns = []

    for _ in range(episodes):
        state = env.reset()
        done = False
        ep_return = 0

        while not done:
            state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            logits = agent.policy(state_t)
            probs = torch.softmax(logits, dim=-1)
            action = torch.argmax(probs, dim=-1).item()

            next_state, reward, done, _ = env.step(action)
            ep_return += reward
            state = next_state

        returns.append(ep_return)

    return returns


In [ ]:
returns = evaluate(agent, episodes=50)
returns[:10]


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(returns, label="Episode Return")
plt.title("RL Autoscaling Agent Evaluation")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
print("Average return:", np.mean(returns))
print("Max return:", np.max(returns))
print("Min return:", np.min(returns))


In [ ]:
env = AutoscaleEnv()
state = env.reset()

for i in range(10):
    state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    logits = agent.policy(state_t)
    probs = torch.softmax(logits, dim=-1)
    action = torch.argmax(probs, dim=-1).item()

    next_state, reward, done, _ = env.step(action)

    print(f"Step {i}: action={action}, reward={reward:.4f}, replicas={env.replicas}")

    state = next_state
